---
 
# POWER BI OPERATIONAL METRIC DAX GENERATOR
---
## Generates:
 1. Metric layout DATATABLE
 2. Placeholder variables
 3. Metric-specific status variables
 4. Dynamic status mapping
 5. Dynamic YTD mapping
 6. Dynamic prior-month mappings
 7. Dynamic current-month mapping
 8. Dynamic format-string mapping
 9. Complete _operationalRows HTML block
 10. Text-file export
 11. Automatic clipboard copy

# Edit only the CONFIGURATION section for a new template.
---

---
# METRIC METADATA
---
Required fields:
 1. SortOrder
 2. MetricName
 3. MetricKey
 4. FormatString
 5. ThresholdFlag
 6. RedOperator
 7. RedThreshold
 8. AmberOperator
 9. AmberThreshold

Status rules are evaluated from top to bottom:
  1. Red condition
  2. Amber condition
  3. Otherwise green

Examples:

Lower values are bad:
  RedOperator: "<"
  RedThreshold: 0.75
  AmberOperator: "<"
  AmberThreshold: 1.00

Higher values are bad:
  RedOperator: ">"
  RedThreshold: 0.25
  AmberOperator: ">="
  AmberThreshold: 0.20

DAX formatting examples:
  Percentage: "0.0%"
  Whole number: "#,##0"
  One decimal: "#,##0.0"
  Currency: "$#,##0"
  Currency with cents: "$#,##0.00"


In [6]:
from pathlib import Path
from typing import Any
import pandas as pd

# ============================================================
# RETRIEVE METRIC DEFINITIONS
# ============================================================

metrics_df = pd.read_excel(
    r"C:\Users\StevenFoster\APM US\Data and Insights - Documents\Power BI Source Files\Balanced Scorecard\Operational Excellence Metric Thresholds.xlsx",
    engine="openpyxl"
)
# Convert to dict
Metrics = metrics_df.to_dict(orient="records")
# ============================================================
# CONFIGURATION
# ============================================================

LayoutTableName = "WeCare Operational Metric Layout"

OutputFileName = "WeCare_Operational_Metric_DAX.txt"

CopyToClipboard = True

GenerateLayoutTable = True

# If True, a blank current-month value returns BLANK() for status.
# This prevents missing metrics from incorrectly displaying green.
BlankStatusWhenValueIsBlank = True

# Controls indentation of the generated DAX.
BaseIndent = 0

# ============================================================
# VALIDATION SETTINGS
# ============================================================

ValidOperators = {"<", "<=", ">", ">=", "=", "<>"}

RequiredFields = {
    "SortOrder",
    "MetricName",
    "MetricKey",
    "FormatString",
    "ThresholdFlag",
    "RedOperator",
    "RedThreshold",
    "AmberOperator",
    "AmberThreshold",
    "YTDFlag",
    "YoYFlag"
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def ValidateMetrics(MetricList: list[dict[str, Any]]) -> None:
    """
    Validates metric metadata before generating DAX.

    Raises a clear exception when:
      - The metric list is empty.
      - A required field is missing.
      - Metric keys are duplicated.
      - Sort orders are duplicated.
      - Operators are unsupported.
      - Thresholds are not numeric.
    """

    if not MetricList:
        raise ValueError("Metrics cannot be empty.")

    MetricKeys = []
    SortOrders = []

    for MetricIndex, Metric in enumerate(MetricList, start=1):
        MissingFields = RequiredFields.difference(Metric.keys())

        if MissingFields:
            MissingText = ", ".join(sorted(MissingFields))
            raise ValueError(
                f"Metric {MetricIndex} is missing required fields: "
                f"{MissingText}"
            )

        MetricName = str(Metric["MetricName"]).strip()
        MetricKey = str(Metric["MetricKey"]).strip()
        FormatString = str(Metric["FormatString"]).strip()
        ThresholdFlag = str(Metric["ThresholdFlag"]).strip().lower()

        if not MetricName:
            raise ValueError(
                f"Metric {MetricIndex} has a blank MetricName."
            )

        if not MetricKey:
            raise ValueError(
                f"Metric {MetricIndex} has a blank MetricKey."
            )

        if not MetricKey.replace("_", "").isalnum():
            raise ValueError(
                f"MetricKey '{MetricKey}' contains unsupported characters. "
                "Use letters, numbers, or underscores only."
            )

        if not FormatString:
            raise ValueError(
                f"Metric '{MetricName}' has a blank FormatString."
            )
        
        if ThresholdFlag not in {"yes", "no"}:
            raise ValueError(
                f"Metric '{MetricName}' has an invalid ThresholdFlag. "
                "Use Yes or No."
            )
        if not isinstance(Metric["RedThreshold"], (int, float)):
            raise TypeError(
                f"RedThreshold for '{MetricName}' must be numeric."
            )

        if not isinstance(Metric["AmberThreshold"], (int, float)):
            raise TypeError(
                f"AmberThreshold for '{MetricName}' must be numeric."
            )

        # if not isinstance(Metric["SortOrder"], int):
        #     raise TypeError(
        #         f"SortOrder for '{MetricName}' must be an integer."
        #     )

        MetricKeys.append(MetricKey)
        # SortOrders.append(Metric["SortOrder"])

    DuplicateMetricKeys = {
        MetricKey
        for MetricKey in MetricKeys
        if MetricKeys.count(MetricKey) > 1
    }

    if DuplicateMetricKeys:
        DuplicateText = ", ".join(sorted(DuplicateMetricKeys))
        raise ValueError(
            f"Duplicate MetricKey values found: {DuplicateText}"
        )

    DuplicateSortOrders = {
        SortOrder
        for SortOrder in SortOrders
        if SortOrders.count(SortOrder) > 1
    }

    if DuplicateSortOrders:
        DuplicateText = ", ".join(
            str(SortOrder)
            for SortOrder in sorted(DuplicateSortOrders)
        )
        raise ValueError(
            f"Duplicate SortOrder values found: {DuplicateText}"
        )


def GetVariableName(MetricKey: str) -> str:
    """
    Converts a MetricKey into the Power BI variable naming pattern.

    Example:
        CoDeterminations -> coDeterminations
        AdultMSG        -> adultMSG
    """

    CleanMetricKey = MetricKey.strip()

    return CleanMetricKey[0].lower() + CleanMetricKey[1:]


def EscapeDaxText(Value: str) -> str:
    """
    Escapes double quotation marks for DAX string literals.
    """

    return str(Value).replace('"', '""')


def FormatDaxNumber(Value: int | float) -> str:
    """
    Produces a clean DAX-compatible numeric literal.

    Examples:
        1.0  -> 1
        0.75 -> 0.75
        30   -> 30
    """

    if isinstance(Value, bool):
        raise TypeError("Boolean values cannot be used as thresholds.")

    if isinstance(Value, int):
        return str(Value)

    FormattedValue = format(Value, ".15g")

    if FormattedValue.startswith("0."):
        return FormattedValue

    if FormattedValue.startswith("-0."):
        return FormattedValue

    return FormattedValue


def GetIndentedText(Text: str, IndentLevel: int) -> str:
    """
    Adds a consistent number of leading spaces to every nonblank line.
    """

    Prefix = " " * IndentLevel

    return "\n".join(
        f"{Prefix}{Line}" if Line else ""
        for Line in Text.splitlines()
    )


def BuildSectionHeader(SectionName: str) -> str:
    """
    Creates a consistent DAX section header.
    """

    return (
        "/* ============================================================\n"
        f"   {SectionName}\n"
        "   ============================================================ */"
    )


# ============================================================
# DAX LAYOUT TABLE GENERATOR
# ============================================================

def BuildLayoutTable(
    TableName: str,
    MetricList: list[dict[str, Any]],
) -> str:
    """
    Generates the full DAX DATATABLE for the metric layout.
    """

    SortedMetrics = sorted(
        MetricList,
        key=lambda Metric: Metric["SortOrder"],
    )

    TableRows = []

    for Metric in SortedMetrics:
        MetricName = EscapeDaxText(Metric["MetricName"])
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        SortOrder = Metric["SortOrder"]

        TableRows.append(
            f'    {{{SortOrder}, "{MetricName}", "{MetricKey}"}}'
        )

    RowText = ",\n".join(TableRows)

    return (
        f"{TableName} =\n"
        "DATATABLE(\n"
        '    "SortOrder", INTEGER,\n'
        '    "MetricName", STRING,\n'
        '    "MetricKey", STRING,\n'
        "{\n"
        f"{RowText}\n"
        "}\n"
        ")"
    )


# ============================================================
# PLACEHOLDER VARIABLE GENERATOR
# ============================================================

def BuildPlaceholderVariables(
    MetricList: list[dict[str, Any]],
) -> str:
    """
    Generates placeholder variables based on YTDFlag and YoYFlag.
    """

    PlaceholderBlocks = [
        BuildSectionHeader("OPERATIONAL EXCELLENCE PLACEHOLDERS")
    ]

    for Metric in MetricList:
        MetricName = Metric["MetricName"]
        VariableName = GetVariableName(Metric["MetricKey"])

        YTDFlag = str(Metric.get("YTDFlag", "")).strip().lower() == "yes"
        YoYFlag = str(Metric.get("YoYFlag", "")).strip().lower() == "yes"


        VariableLines = [
            (
                f"VAR _{VariableName} = 1 "
                f"// {MetricName}"
            ),
            (
                f"VAR _{VariableName}Month3 = "
                f"CALCULATE(_{VariableName},ALL(dimdate),"
                f"dimdate[Month_Year] = _month3)"
            ),
            (
                f"VAR _{VariableName}Month2 = "
                f"CALCULATE(_{VariableName},ALL(dimdate),"
                f"dimdate[Month_Year] = _month2)"
            ),
            (
                f"VAR _{VariableName}Month = "
                f"CALCULATE(_{VariableName},ALL(dimdate),"
                f"dimdate[Month_Year] = _month1)"
            ),
        ]

        if YTDFlag:
            VariableLines.append(
                (
                    f"VAR _{VariableName}Ytd = "
                    f"CALCULATE(_{VariableName},ALL(dimdate),"
                    f"dimdate[Fiscal Year] = _year,"
                    f"dimdate[Month Sort] <= _currentmonthsort)"
                )
            )

        if YoYFlag:
            VariableLines.extend(
                [
                    (
                        f"VAR _{VariableName}Pytd = "
                        f"CALCULATE(_{VariableName},ALL(dimdate),"
                        f"dimdate[Fiscal Year] = _priorYear,"
                        f"dimdate[Month Sort] <= _currentmonthsort)"
                    ),
                    (
                        f"VAR _{VariableName}Var = "
                        f"_{VariableName}Ytd - _{VariableName}Pytd"
                    ),
                    (
                        f"VAR _{VariableName}VariancePct = "
                        f"DIVIDE(_{VariableName}Var,"
                        f"_{VariableName}Pytd)"
                    ),
                    (
                        f"VAR _{VariableName}Trend = "
                        f"SWITCH( TRUE(), ISBLANK(_{VariableName}Var), BLANK(),"
                        f' _{VariableName}Var > 0, "up", '
                        f' _{VariableName}Var < 0, "down", '
                        f" BLANK() )"
                    ),
                ]
            )

        PlaceholderBlocks.append("\n".join(VariableLines))

    return "\n\n".join(PlaceholderBlocks)


# ============================================================
# STATUS VARIABLE GENERATOR
# ============================================================

def BuildStatusVariables(
    MetricList: list[dict[str, Any]],
    ReturnBlankStatus: bool,
) -> str:
    """
    Generates metric-specific status variables using each metric's
    operators and thresholds.
    """

    StatusBlocks = [
        BuildSectionHeader("OPERATIONAL EXCELLENCE STATUS")
    ]
    for Metric in MetricList:
    
        VariableName = GetVariableName(
            Metric["MetricKey"]
        )
    
        ThresholdFlag = str(
            Metric["ThresholdFlag"]
        ).strip().lower()
    
        #
        # Metrics without thresholds
        #
        if ThresholdFlag == "no":
    
            StatusBlocks.append(
                f"VAR _{VariableName}Status = BLANK()"
            )
    
            continue
    
        RedOperator = Metric["RedOperator"]
        RedThreshold = FormatDaxNumber(
            Metric["RedThreshold"]
        )
    
        AmberOperator = Metric["AmberOperator"]
        AmberThreshold = FormatDaxNumber(
            Metric["AmberThreshold"]
        )
        StatusLines = [
            f"VAR _{VariableName}Status =",
            "    SWITCH(",
            "        TRUE(),",
        ]

        if ReturnBlankStatus:
            StatusLines.append(
                f"        ISBLANK(_{VariableName}Month), BLANK(),"
            )

        StatusLines.extend(
            [
                (
                    f"        _{VariableName}Month "
                    f"{RedOperator} {RedThreshold}, \"red\","
                ),
                (
                    f"        _{VariableName}Month "
                    f"{AmberOperator} {AmberThreshold}, \"amber\","
                ),
                '        "green"',
                "    )",
            ]
        )

        StatusBlocks.append("\n".join(StatusLines))

    return "\n\n".join(StatusBlocks)


# ============================================================
# SWITCH GENERATOR
# ============================================================

def BuildMetricSwitch(
    VariableName: str,
    MetricList: list[dict[str, Any]],
    ValueSuffix: str,
    IndentLevel: int = 4,
) -> str:
    """
    Generates a SWITCH statement mapping MetricKey to a metric
    variable.

    Example:
        VAR _ytd =
            SWITCH(
                _metricKey,
                "MetricKey", _metricYtd
            )
    """

    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)

    SwitchLines = [
        f"{Indent}VAR _{VariableName} =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []

    for Metric in MetricList:
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        MetricVariableName = GetVariableName(
            Metric["MetricKey"]
        )

        MetricLines.append(
            f'{ValueIndent}"{MetricKey}", '
            f"_{MetricVariableName}{ValueSuffix}"
        )

    for MetricIndex, MetricLine in enumerate(MetricLines):
        IsLastMetric = MetricIndex == len(MetricLines) - 1
        LineEnding = "" if IsLastMetric else ","

        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")

    return "\n".join(SwitchLines)


def BuildFormatSwitch(
    MetricList: list[dict[str, Any]],
    IndentLevel: int = 4,
) -> str:
    """
    Generates the dynamic DAX format-string mapping.
    """

    Indent = " " * IndentLevel
    InnerIndent = " " * (IndentLevel + 4)
    ValueIndent = " " * (IndentLevel + 8)

    SwitchLines = [
        f"{Indent}VAR _formatString =",
        f"{InnerIndent}SWITCH(",
        f"{ValueIndent}_metricKey,",
    ]

    MetricLines = []

    for Metric in MetricList:
        MetricKey = EscapeDaxText(Metric["MetricKey"])
        FormatString = EscapeDaxText(
            Metric["FormatString"]
        )

        MetricLines.append(
            f'{ValueIndent}"{MetricKey}", "{FormatString}"'
        )

    for MetricIndex, MetricLine in enumerate(MetricLines):
        IsLastMetric = MetricIndex == len(MetricLines) - 1
        LineEnding = "" if IsLastMetric else ","

        SwitchLines.append(f"{MetricLine}{LineEnding}")

    SwitchLines.append(f"{InnerIndent})")

    return "\n".join(SwitchLines)


# ============================================================
# DYNAMIC HTML ROW GENERATOR
# ============================================================

def BuildOperationalRows(
    TableName: str,
    MetricList: list[dict[str, Any]],
) -> str:
    """
    Generates the complete _operationalRows CONCATENATEX block.

    The monthly values are mapped and available for future HTML
    columns, even though the current row HTML only displays YTD.
    """

    StatusSwitch = BuildMetricSwitch(
        VariableName="status",
        MetricList=MetricList,
        ValueSuffix="Status",
        IndentLevel=4,
    )

    YtdMetrics = [
        Metric
        for Metric in MetricList
        if str(Metric.get("YTDFlag", "")).strip().lower() == "yes"
    ]
    
    YtdSwitch = BuildMetricSwitch(
        VariableName="ytd",
        MetricList=YtdMetrics,
        ValueSuffix="Ytd",
        IndentLevel=4,
    )

    MonthPrior2Switch = BuildMetricSwitch(
        VariableName="monthPrior2",
        MetricList=MetricList,
        ValueSuffix="Month3",
        IndentLevel=4,
    )

    MonthPrior1Switch = BuildMetricSwitch(
        VariableName="monthPrior1",
        MetricList=MetricList,
        ValueSuffix="Month2",
        IndentLevel=4,
    )

    CurrentMonthSwitch = BuildMetricSwitch(
        VariableName="currentMonth",
        MetricList=MetricList,
        ValueSuffix="Month",
        IndentLevel=4,
    )

    YoYMetrics = [
        Metric
        for Metric in MetricList
        if str(Metric.get("YoYFlag", "")).strip().lower() == "yes"
    ]

    TrendSwitch = ""
    VarianceSwitch = ""
    
    if YoYMetrics:
        TrendSwitch = BuildMetricSwitch(
            VariableName="trendIcon",
            MetricList=YoYMetrics,
            ValueSuffix="Trend",
            IndentLevel=4,
        )
    
        VarianceSwitch = BuildMetricSwitch(
            VariableName="pyVAR",
            MetricList=YoYMetrics,
            ValueSuffix="VariancePct",
            IndentLevel=4,
        )
    FormatSwitch = BuildFormatSwitch(
        MetricList=MetricList,
        IndentLevel=4,
    )
    
    TrendHtml = '        VAR _trendHtml = ""\n'
    if YoYMetrics:
        TrendHtml = (
            "        VAR _trendHtml =\n"
            "            SWITCH(\n"
            "                TRUE(),\n"
            "                ISBLANK(_pyVAR), \"\",\n"
            "                _trendIcon = \"up\",\n"
            "                    \"<span class='trend up'>▲ \"\n"
            "                        & FORMAT(_pyVAR, \"0.0%\")\n"
            "                        & \"</span>\",\n"
            "                _trendIcon = \"down\",\n"
            "                    \"<span class='trend down'>▼ \"\n"
            "                        & FORMAT(ABS(_pyVAR), \"0.0%\")\n"
            "                        & \"</span>\",\n"
            "                FORMAT(_pyVAR, \"0.0%\")\n"
            "            )\n"
        )


    return (
        f"{BuildSectionHeader('DYNAMIC HTML CREATION')}\n"
        "VAR _operationalRows =\n"
        "    CONCATENATEX(\n"
        f"        '{EscapeDaxText(TableName)}',\n"
        "        VAR _metricKey = [MetricKey]\n"
        "\n"
        f"{StatusSwitch}\n"
        "\n"
        f"{YtdSwitch}\n"
        "\n"
        f"{MonthPrior2Switch}\n"
        "\n"
        f"{MonthPrior1Switch}\n"
        "\n"
        f"{CurrentMonthSwitch}\n" 
        "\n"
        f"{TrendSwitch}\n"     
        "\n" 
        f"{VarianceSwitch}\n"
        "\n"
        f"{TrendHtml}\n" 
        "\n"
        f"{FormatSwitch}\n"
        "\n"
        "        RETURN\n"
        '            "<tr>" &\n'
        '            "<td class=''label''>" & [MetricName] & "</td>" &\n'
        "            \"<td class='status'>\" &\n"
        "                \"<span class='dot \" &\n"
        "                    COALESCE(_status, \"\") &\n"
        "                \"'></span>\" &\n"
        '            "</td>" &\n'
        "            \"<td class='number'>\" &\n"
        "                IF(\n"
        "                    ISBLANK(_ytd),\n"
        '                    "",\n'
        "                    FORMAT(_ytd, _formatString)\n"
        "                ) &\n"
        '                "</td>"\n'
        '                & "<td class=\'number\'>"\n'
        "                & IF\n"
        "                (\n"
        "                    ISBLANK ( _monthPrior2 ),\n"
        '                    "",\n'
        "                    FORMAT ( _monthPrior2, _formatString )\n"
        "                )\n"
        '                & "</td>"\n'
        '                & "<td class=\'number\'>"\n'
        "                & IF\n"
        "                (\n"
        "                    ISBLANK ( _monthPrior1 ),\n"
        '                    "",\n'
        "                    FORMAT ( _monthPrior1, _formatString )\n"
        "                )\n"
        '                & "</td>"\n'
        '                & "<td class=\'number\'>"\n'
        "                & IF\n"
        "                (\n"
        "                    ISBLANK ( _currentMonth ),\n"
        '                    "",\n'
        "                    FORMAT ( _currentMonth, _formatString )\n"
        "                )\n"
        '                & "</td>" &\n'
       '                "<td class=\'number\'>"\n'
        "                & IF\n"
        "                (\n"
        "                    ISBLANK ( _trendHtml ),\n"
        '                    "",\n'
        "                    _trendHtml\n"
        "                )\n"
        '                & "</td>" &\n'
        '            "</td>" &\n'
        '            "</tr>",\n'
        '        "",\n'
        "        [SortOrder],\n"
        "        ASC\n"
        "    )"
        "\n\n"
        "RETURN\n"
        '    "\n'
        '    <!-- OPERATIONAL EXCELLENCE -->\n'
        '    <tr>\n'
        '      <td colspan=\'6\' class=\'header\'>\n'
        '        Operational Excellence\n'
        '      </td>\n'
        '    </tr>\n'
        '    <tr>\n'
        '      <td class=\'left\'>\n'
        '        <table class=\'metric\'>\n'
        '          <colgroup>\n'
        '            <col style=\'width:34%\'>\n'
        '            <col style=\'width:3%\'>\n'
        '            <col style=\'width:12.6%\'>\n'
        '            <col style=\'width:12.6%\'>\n'
        '            <col style=\'width:12.6%\'>\n'
        '            <col style=\'width:12.6%\'>\n'
        '            <col style=\'width:12.6%\'>\n'
        '          </colgroup>\n'
        '\n'
        '          <tr class=\'subheader\'>\n'
        '            <th></th>\n'
        '            <th></th>\n'
        '            <th>PY \'" & FORMAT(RIGHT(_year,2),"##") & " YTD</th>\n'
        '            <th>" & _displayMonth3 & "</th>\n'
        '            <th>" & _displayMonth2 & "</th>\n'
        '            <th>" & _displayMonth1 & "</th>\n'
        '            <th>" & _trending & "</th>\n'
        '          </tr>" &\n'
        '\n'
        '    _operationalRows &\n'
        '\n'
        '    "\n'
        '        </table>\n'
        '      </td>\n'
        '      <td class=\'right commentbackground\'>\n'
        '        <div class=\'commentsheader\'></div>\n'
        '        <div class=\'commentary\'></div>\n'
        '      </td>\n'
        '    </tr>\n'
        '    "'
    )


# ============================================================
# FULL DAX GENERATOR
# ============================================================

def GenerateDax(
    TableName: str,
    MetricList: list[dict[str, Any]],
    IncludeLayoutTable: bool = True,
    ReturnBlankStatus: bool = True,
    IndentLevel: int = 0,
) -> str:
    """
    Validates the metadata and generates the complete DAX output.
    """

    ValidateMetrics(MetricList)

    SortedMetrics = sorted(
        MetricList,
        key=lambda Metric: Metric["SortOrder"],
    )

    OutputSections = []

    if IncludeLayoutTable:
        OutputSections.extend(
            [
                BuildSectionHeader("METRIC LAYOUT TABLE"),
                BuildLayoutTable(
                    TableName=TableName,
                    MetricList=SortedMetrics,
                ),
            ]
        )

    OutputSections.extend(
        [
            BuildSectionHeader(
                "TREND COLUMN VARIABLE"
            ),
            'VAR _trending = "YoY Trend"',
    
            BuildSectionHeader(
                "TIME PLACEHOLDERS"
            ),
            "VAR _year = [Current Fiscal Year]",
            "VAR _priorYear = [Prior Fiscal YTD]",
            "VAR _currentmonthsort = [Current Month Sort]",
            "VAR _month3 = [Month Year Prior 2]",
            "VAR _month2 = [Month Year Prior 1]",
            "VAR _month1 = [Month Year Current]",
            "VAR _displayMonth3 = [Month Display Prior 2]",
            "VAR _displayMonth2 = [Month Display Prior 1]",
            "VAR _displayMonth1 = [Month Display Current]",
    
            BuildPlaceholderVariables(
                MetricList=SortedMetrics,
            ),
    
            BuildStatusVariables(
                MetricList=SortedMetrics,
                ReturnBlankStatus=ReturnBlankStatus,
            ),
    
            BuildOperationalRows(
                TableName=TableName,
                MetricList=SortedMetrics,
            )
        ]
    )

    GeneratedDax = "\n\n".join(OutputSections)

    return GetIndentedText(
        Text=GeneratedDax.strip(),
        IndentLevel=IndentLevel,
    )


# ============================================================
# CLIPBOARD FUNCTION
# ============================================================

def CopyTextToClipboard(Text: str) -> bool:
    """
    Copies text to the system clipboard using pyperclip.

    Returns True when successful and False when clipboard access
    is unavailable.
    """

    try:
        import pyperclip

        pyperclip.copy(Text)

        ClipboardTest = pyperclip.paste()

        if ClipboardTest != Text:
            print(
                "Warning: Clipboard verification did not match "
                "the generated DAX."
            )
            return False

        return True

    except ImportError:
        print(
            "Clipboard copy skipped because pyperclip is not "
            "installed."
        )
        print(
            "Install it in Jupyter with: %pip install pyperclip"
        )
        return False

    except Exception as ClipboardError:
        print(
            "Clipboard copy failed, but the DAX file was still "
            "created."
        )
        print(f"Clipboard error: {ClipboardError}")
        return False


# ============================================================
# GENERATE, SAVE, COPY, AND DISPLAY
# ============================================================

try:
    DaxCode = GenerateDax(
        TableName=LayoutTableName,
        MetricList=Metrics,
        IncludeLayoutTable=GenerateLayoutTable,
        ReturnBlankStatus=BlankStatusWhenValueIsBlank,
        IndentLevel=BaseIndent,
    )

    OutputPath = Path(OutputFileName).resolve()

    OutputPath.write_text(
        DaxCode,
        encoding="utf-8",
    )

    ClipboardCopied = False

    if CopyToClipboard:
        ClipboardCopied = CopyTextToClipboard(DaxCode)

    print("=" * 70)
    print("DAX GENERATION COMPLETE")
    print("=" * 70)
    print(f"Metric count: {len(Metrics)}")
    print(f"Layout table: {LayoutTableName}")
    print(f"Saved file: {OutputPath}")

    if CopyToClipboard:
        if ClipboardCopied:
            print("Clipboard: DAX copied successfully")
        else:
            print("Clipboard: DAX was not copied")
    else:
        print("Clipboard: Disabled in configuration")

    print("=" * 70)
    print()
    print(DaxCode)

except Exception as GenerationError:
    print("=" * 70)
    print("DAX GENERATION FAILED")
    print("=" * 70)
    print(str(GenerationError))
    raise

DAX GENERATION COMPLETE
Metric count: 6
Layout table: WeCare Operational Metric Layout
Saved file: C:\Users\StevenFoster\APM US\Data and Insights - Documents\Python Scripts\Balanced Scorecard\WeCare_Operational_Metric_DAX.txt
Clipboard: DAX copied successfully

/* ============================================================
   METRIC LAYOUT TABLE
   ============================================================ */

WeCare Operational Metric Layout =
DATATABLE(
    "SortOrder", INTEGER,
    "MetricName", STRING,
    "MetricKey", STRING,
{
    {1, "Clinical Assessment Completions", "_completions"},
    {2, "SSI/SSDI Applications Submitted", "_submitted"},
    {3, "VRS Initiations", "_var"},
    {4, "Day 1 Placements (% Target)", "_placements"},
    {5, "30-Day Employment Retention (% Target)", "_employment"},
    {6, "90-Day Employment Retention (% Target)", "_retention"}
}
)

/* ============================================================
   TREND COLUMN VARIABLE
   ======================